In [ ]:

!pip install crewai langchain-google-genai tavily-python pydantic pyyaml --quiet

import os
import yaml
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure API keys and model
class Config:
    def __init__(self, model_name: str = "gemini-pro"):
        self.api_key = os.getenv("GEMINI_API_KEY", "your_gemini_api_key")
        self.tavily_key = os.getenv("TAVILY_API_KEY", "your_tavily_api_key")
        if not self.api_key or self.api_key == "your_gemini_api_key":
            raise EnvironmentError("GEMINI_API_KEY not set")
        if not self.tavily_key or self.tavily_key == "your_tavily_api_key":
            raise EnvironmentError("TAVILY_API_KEY not set")
        self.model_name = model_name
        self.tavily_client = TavilyClient(api_key=self.tavily_key)

    def get_llm(self):
        return ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model=self.model_name,
            temperature=0.6,  # Changed from 0.7
            max_tokens=1500   # Added token limit
        )

# Data models
class Resource(BaseModel):
    name: str = Field(description="Name of the resource")
    category: str = Field(description="Category: video, article, book, exercise")
    link: str = Field(description="Resource URL")
    summary: str = Field(description="Brief summary of content")
    level: str = Field(description="Level: beginner, intermediate, advanced")

class ResourceList(BaseModel):
    subject: str = Field(description="Subject area")
    resources: List[Resource] = Field(description="Curated resources")

class Question(BaseModel):
    text: str = Field(description="Question text")
    choices: List[str] = Field(description="Answer choices")
    answer: str = Field(description="Correct answer")
    rationale: str = Field(description="Answer explanation")

class Assessment(BaseModel):
    subject: str = Field(description="Assessment subject")
    level: str = Field(description="Difficulty level")
    questions: List[Question] = Field(description="List of questions")

class Project(BaseModel):
    name: str = Field(description="Project name")
    overview: str = Field(description="Project overview")
    skills: List[str] = Field(description="Required skills")
    duration: str = Field(description="Estimated duration")
    level: str = Field(description="Project difficulty")
    tools: List[str] = Field(description="Required tools")

class ProjectList(BaseModel):
    subject: str = Field(description="Subject area")
    skill_level: str = Field(description="User's skill level")
    projects: List[Project] = Field(description="Project suggestions")

# Tools
@tool("project_recommendation")
def project_recommendation(subject: str, skill_level: str) -> str:
    """Generate project ideas based on subject and skill level."""
    project_db = {
        "novice": {
            "coding": [
                "Simple text-based game",
                "Basic task tracker",
                "Personal webpage"
            ],
            "data analysis": [
                "Visualize open dataset",
                "Basic statistical analysis",
                "Simple forecasting model"
            ],
            "web design": [
                "Portfolio site",
                "Static blog page",
                "Mock product landing"
            ]
        },
        "moderate": {
            "coding": [
                "RESTful API service",
                "Cross-platform app",
                "Database-driven tool"
            ],
            "data analysis": [
                "ML model implementation",
                "Data dashboard",
                "Recommendation engine"
            ],
            "web design": [
                "Full-stack website",
                "E-commerce platform",
                "Content manager"
            ]
        },
        "expert": {
            "coding": [
                "Distributed app framework",
                "Custom language parser",
                "Game development kit"
            ],
            "data analysis": [
                "Deep neural network",
                "Real-time data pipeline",
                "Image recognition system"
            ],
            "web design": [
                "Microservices web app",
                "Progressive web application",
                "Collaborative platform"
            ]
        }
    }

    subject_lower = subject.lower()
    level_lower = skill_level.lower()
    projects = []
    for category, items in project_db.get(level_lower, {}).items():
        if category in subject_lower or subject_lower in category:
            projects.extend(items)
    if not projects:
        projects = project_db.get(level_lower, {}).get("coding", [])

    return f"Projects for {subject} ({skill_level}): " + ", ".join(projects)

@tool("resource_search")
def resource_search(query: str) -> str:
    """Search for educational resources using Tavily."""
    try:
        config = Config()
        results = config.tavily_client.search(
            query=query,
            search_depth="advanced",  # Changed from basic
            max_results=3            # Changed from 5
        )
        formatted = []
        for r in results.get('results', []):
            formatted.extend([
                f"Name: {r.get('title', 'Unknown')}",
                f"Link: {r.get('url', 'Unknown')}",
                f"Summary: {r.get('content', 'No content')[:150]}...",
                "---"
            ])
        return "\n".join(formatted)
    except Exception as e:
        return f"Search failed: {str(e)}"

# Agents
resource_finder = Agent(
    role="Resource Specialist",
    goal="Find high-quality educational resources for specific subjects",
    backstory="An expert in sourcing educational content across various formats and platforms.",
    tools=[resource_search],
    verbose=True,
    llm=Config().get_llm()
)

assessment_designer = Agent(
    role="Assessment Developer",
    goal="Create tailored assessments to evaluate learner understanding",
    backstory="A proficient educator skilled in designing effective assessments.",
    verbose=True,
    llm=Config().get_llm()
)

project_advisor = Agent(
    role="Project Mentor",
    goal="Suggest relevant projects to enhance practical skills",
    backstory="A guide for learners, providing actionable project ideas to apply knowledge.",
    tools=[project_recommendation],
    verbose=True,
    llm=Config().get_llm()
)

# Tasks
def create_resource_task(subject: str, skill_level: str) -> Task:
    return Task(
        description=f"""
        Find educational resources for {subject} at {skill_level} level.
        Include:
        1. Videos from trusted platforms
        2. Articles or guides
        3. Books or references
        Provide 3 resources suitable for {skill_level} learners.
        """,
        expected_output="A list of educational resources with names, categories, links, summaries, and levels",
        agent=resource_finder,
        output_pydantic=ResourceList
    )

def create_assessment_task(subject: str, skill_level: str) -> Task:
    return Task(
        description=f"""
        Design an assessment for {subject} at {skill_level} level.
        Create 3 multiple-choice questions that:
        1. Cover core concepts
        2. Suit {skill_level} learners
        3. Include 3 options each
        4. Provide answer explanations
        """,
        expected_output="An assessment with 3 questions, options, answers, and explanations",
        agent=assessment_designer,
        output_pydantic=Assessment
    )

def create_project_task(subject: str, skill_level: str) -> Task:
    return Task(
        description=f"""
        Recommend projects for {subject} at {skill_level} level.
        Provide:
        1. 2-3 project ideas
        2. Detailed overviews
        3. Skills and tools needed
        4. Estimated timelines
        Projects should be practical and reinforce learning.
        """,
        expected_output="A list of projects with details, skills, and timelines",
        agent=project_advisor,
        output_pydantic=ProjectList
    )

# Main execution
def run_education_system():
    print("=== Educational Resource System ===")
    subject = input("Enter subject of interest: ").strip()
    skill_level = input("Enter skill level (novice/moderate/expert): ").strip().lower()

    if skill_level not in ["novice", "moderate", "expert"]:
        skill_level = "novice"
        print("Invalid skill level. Defaulting to 'novice'.")

    print(f"\n🔍 Generating resources for: {subject} ({skill_level})")
    print("Processing...")

    resource_task = create_resource_task(subject, skill_level)
    assessment_task = create_assessment_task(subject, skill_level)
    project_task = create_project_task(subject, skill_level)

    crew = Crew(
        agents=[resource_finder, assessment_designer, project_advisor],
        tasks=[resource_task, assessment_task, project_task],
        process=Process.sequential,
        verbose=True
    )

    try:
        results = crew.kickoff()
        output = {
            "resources": resource_task.output.raw,
            "assessment": assessment_task.output.raw,
            "projects": project_task.output.raw
        }

        # Save output as YAML
        with open("education_output.yaml", "w") as f:
            yaml.dump(output, f, allow_unicode=True)

        print("\n" + "="*50)
        print("✅ Resources Generated!")
        print("="*50)
        print("\n📖 Resources: Check education_output.yaml for details.")
        print("\n📋 Assessment: Custom questions generated.")
        print("\n💡 Projects: Practical project ideas suggested.")
        print(f"\n🎯 Tailored for {skill_level} learners.")
        print("Output saved to education_output.yaml")

        return output

    except Exception as e:
        print(f"System error: {str(e)}")
        return None

def quick_test():
    print("=== Quick Test Mode ===")
    subject = "Data Analysis"
    skill_level = "novice"
    print(f"Testing: {subject} ({skill_level})")

    projects = project_recommendation(subject, skill_level)
    print(f"\nProjects: {projects}")

    resources = resource_search(f"{subject} resources for {skill_level}")
    print(f"\nResources: {resources[:150]}...")

if __name__ == "__main__":
    config = Config()
    try:
        choice = input("Select mode (1: Full System, 2: Quick Test): ").strip()
        if choice == "2":
            quick_test()
        else:
            run_education_system()
    except KeyboardInterrupt:
        print("\nSystem interrupted.")
    except Exception as e:
        print(f"Error: {str(e)}")
        print("Running quick test...")
        quick_test()